In [ ]:
!pip install torchvision
!pip install qwen_vl_utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 21.0 MB/s eta 0:00:00


In [ ]:
from PIL import Image
import numpy as np
from torchvision import transforms
from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info

# default: Load the model on the available device(s)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct", dtype="auto", device_map="auto"
)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")




/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.53G [00:00<?, ?B/s]

... (5 outputs left out)

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

In [ ]:
normalize = transforms.Normalize(
    (0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711))
transform = transforms.Compose([
    transforms.Resize((224,224), interpolation=Image.BICUBIC),
    transforms.ToTensor(),
    normalize,
])
test_img_path = "/content/image1.jpg"
image_pil = Image.open(test_img_path).convert('RGB').resize((224,224))
print(np.array(image_pil).shape)
image = transform(image_pil).unsqueeze(0)
print(image.shape)
sentence = '''"When the plaintiff Kibahan told the above things to his father, the father explained to do so. The said opponent became fiery and started threatening to kill the plaintiff's father. When the plaintiff's father removed Kiran from here, the said opponent came home and repeatedly threatened to kill him. The latest thing is that since 7 pm on the date of 29.11.2020, the whereabouts of the plaintiff's father is not known. The said opponent is repeatedly calling and saying that there will be worse consequences. The plaintiff is also threatened to be killed by kidnapping. The plaintiff's First Information Report was registered against him. It has been submitted by the learned counsel on behalf of the applicant / accused that the applicant / accused is innocent, falsely implicated. The so-called is concocted and concocted. The accused is not remotely connected with the so-called crime. The applicant / accused has no connection with the accused. The family of the applicant / accused was married to Munna, a resident of Ghazipur. The accused had come to their house as a driver to drop his family off. At the same time, the police the applicant / accused with his name. The accused is innocent. The accused is not named in the First Information Report. The accused is a resident of Ghazipur. There is no independent to the arrest. There is no to the incident. The bail application was opposed by the and it is submitted that the name of the applicant / accused has come to light in the course of leading to the incident." '''
prompt = f"""You are a legal expert. Based on the following evidence description, determine if bail should be granted.
Evidence: "{sentence}"
Consider factors like flight risk, danger to community, and evidence strength.
Answer with only 'yes' (bail should be granted) or 'no' (bail should be denied):"""
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": image_pil,
            },
            {"type": "text", "text": prompt},
        ],
    }
]
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
print(text)
image_inputs, video_inputs = process_vision_info(messages)
print(image_inputs)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")
print(inputs)
# Inference: Generation of the output


(224, 224, 3)
torch.Size([1, 3, 224, 224])
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>You are a legal expert. Based on the following evidence description, determine if bail should be granted.
Evidence: ""When the plaintiff Kibahan told the above things to his father, the father explained to do so. The said opponent became fiery and started threatening to kill the plaintiff's father. When the plaintiff's father removed Kiran from here, the said opponent came home and repeatedly threatened to kill him. The latest thing is that since 7 pm on the date of 29.11.2020, the whereabouts of the plaintiff's father is not known. The said opponent is repeatedly calling and saying that there will be worse consequences. The plaintiff is also threatened to be killed by kidnapping. The plaintiff's First Information Report was registered against him. It has been submitted by the learned counsel on behalf of the applicant / accuse

In [ ]:
print(image_inputs[0].size )
lita = image_inputs[0].size

(224, 224)


In [ ]:
generated_ids = model.generate(**inputs, return_dict_in_generate=True,output_scores=True,do_sample=False,       # Greedy decoding
    max_new_tokens=1,temperature=0.1)
print(generated_ids)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


GenerateDecoderOnlyOutput(sequences=tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198, 151652, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151653,   2610,
            525,    264,   5777,   6203,     13,  20205,    389,    279,   2701,
           5904,   4008,     11,   8253,    421,  24479,   1265,    387,  11676,
            624,  92602,     25,   1591,   4498,    279,  42002,    730, 

In [ ]:
import torch
yes_id = processor.tokenizer.encode(" yes", add_special_tokens=False)[0]
no_id = processor.tokenizer.encode(" no", add_special_tokens=False)[0]

# The model generates: "\n" (token 0) then "yes"/"no" (token 1)
# So we need to look at scores[1], not scores[0]
step1_probs = torch.softmax(generated_ids.scores[0][0], dim=0)
yes_prob = step1_probs[yes_id].item()
no_prob = step1_probs[no_id].item()

print(f"Yes probability: {yes_prob}")
print(f"No probability: {no_prob}")
print(f"Yes ID: {yes_id}, No ID: {no_id}")

# Optional: Check what token was actually generated at each step
decoded_0 = processor.tokenizer.decode([generated_ids.sequences[0][-2]])
decoded_1 = processor.tokenizer.decode([generated_ids.sequences[0][-1]])
print(f"Second-to-last token: '{decoded_0}'")
print(f"Last generated token: '{decoded_1}'")

Yes probability: 1.3409082384896465e-05
No probability: 1.3449064681481104e-05
Yes ID: 9834, No ID: 902
Second-to-last token: '
'
Last generated token: 'No'


In [ ]:
# Get all possible token IDs for yes/no
yes_tokens = []
no_tokens = []

for variant in ["yes", " yes", "Yes", " Yes", "YES", " YES"]:
    token_ids = processor.tokenizer.encode(variant, add_special_tokens=False)
    if token_ids:
        yes_tokens.append(token_ids[0])

for variant in ["no", " no", "No", " No", "NO", " NO"]:
    token_ids = processor.tokenizer.encode(variant, add_special_tokens=False)
    if token_ids:
        no_tokens.append(token_ids[0])

# Remove duplicates
yes_tokens = list(set(yes_tokens))
no_tokens = list(set(no_tokens))

print(f"Yes token IDs: {yes_tokens}")
print(f"No token IDs: {no_tokens}")

# Now get probabilities for ALL of them
answer_probs = torch.softmax(generated_ids.scores[0][0], dim=0)

yes_prob = sum([answer_probs[tid].item() for tid in yes_tokens])
no_prob = sum([answer_probs[tid].item() for tid in no_tokens])

print(f"Total Yes probability: {yes_prob:.4f}")
print(f"Total No probability: {no_prob:.4f}")

# Verify: what token was actually generated?
generated_token_id = generated_ids.sequences[0][-1].item()
print(f"\nActually generated token ID: {generated_token_id}")
print(f"Decoded: '{processor.tokenizer.decode([generated_token_id])}'")

Yes token IDs: [14080, 9834, 9454, 14004, 7414, 9693]
No token IDs: [5664, 2753, 2308, 8996, 902, 2152]
Total Yes probability: 0.1251
Total No probability: 0.8670

Actually generated token ID: 2753
Decoded: 'No'


In [ ]:
from PIL import Image
import numpy as np
from torchvision import transforms
normalize = transforms.Normalize(
    (0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711))
transform = transforms.Compose([
    transforms.Resize((224,224), interpolation=Image.BICUBIC),
    transforms.ToTensor(),
    normalize,
])
test_img_path = "/content/image1.jpg"
image_pil = Image.open(test_img_path).convert('RGB')
juda = np.array(image_pil)
print(np.array(image_pil))
print(np.array(image_pil).shape)



[[[153 188 254]
  [151 186 252]
  [147 184 252]
  ...
  [133 163 249]
  [132 162 250]
  [132 162 250]]

 [[151 186 252]
  [150 185 251]
  [148 185 253]
  ...
  [135 165 251]
  [135 165 253]
  [134 164 252]]

 [[151 186 252]
  [151 186 252]
  [149 186 254]
  ...
  [137 167 253]
  [136 166 254]
  [136 166 254]]

 ...

 [[212 239 232]
  [207 234 229]
  [211 240 236]
  ...
  [178 216 255]
  [177 215 254]
  [177 215 254]]

 [[214 241 234]
  [208 235 230]
  [212 241 237]
  ...
  [178 218 254]
  [178 216 253]
  [178 216 253]]

 [[215 242 235]
  [209 236 231]
  [212 241 237]
  ...
  [177 217 253]
  [178 216 253]
  [177 215 252]]]
(300, 300, 3)


In [ ]:
p = 16
import copy
image_token_ids = torch.tensor(range(1, p**2+1)).unsqueeze(0) # take one less because CLS and SEP tokens do not count
print(len(inputs['input_ids'][0]))
print(inputs)
X = torch.cat((inputs['input_ids'].cuda(), image_token_ids.cuda()), 1)
print(X)


464
{'input_ids': tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198, 151652, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151653,   2610,
            525,    264,   5777,   6203,     13,  20205,    389,    279,   2701,
           5904,   4008,     11,   8253,    421,  24479,   1265,    387,  11676,
            624,  92602,     25,   1591,   4498,    279,  42002,    730,    579,
          

In [ ]:
copies = X[:,:len(inputs['input_ids'][0])]
copier =  X[:,len(inputs['input_ids'][0]):]
print(copies)
print(copier)
juda = np.array(image_pil)
for i in range(len(copier)):
  if copier[0][i] == 0:
    k = i//16
    l = i%16
    juda[k*14:(k+1)*14,l*14:(l+1)*14,:] = 0
imaged = Image.fromarray(juda)


tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198, 151652, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151653,   2610,
            525,    264,   5777,   6203,     13,  20205,    389,    279,   2701,
           5904,   4008,     11,   8253,    421,  24479,   1265,    387,  11676,
            624,  92602,     25,   1591,   4498,    279,  42002,    730,    579,
          22051,   3229,    

In [ ]:
print(imaged)
print(np.array(imaged))

<PIL.Image.Image image mode=RGB size=300x300 at 0x7CC3EB0E77A0>
[[[153 188 254]
  [151 186 252]
  [147 184 252]
  ...
  [133 163 249]
  [132 162 250]
  [132 162 250]]

 [[151 186 252]
  [150 185 251]
  [148 185 253]
  ...
  [135 165 251]
  [135 165 253]
  [134 164 252]]

 [[151 186 252]
  [151 186 252]
  [149 186 254]
  ...
  [137 167 253]
  [136 166 254]
  [136 166 254]]

 ...

 [[212 239 232]
  [207 234 229]
  [211 240 236]
  ...
  [178 216 255]
  [177 215 254]
  [177 215 254]]

 [[214 241 234]
  [208 235 230]
  [212 241 237]
  ...
  [178 218 254]
  [178 216 253]
  [178 216 253]]

 [[215 242 235]
  [209 236 231]
  [212 241 237]
  ...
  [177 217 253]
  [178 216 253]
  [177 215 252]]]


In [ ]:
# Or without special tokens (cleaner)
print(copies)

modified_text_clean = processor.tokenizer.decode(copies[0].cuda(), skip_special_tokens=True)
print("\nCleaner version:")
print(modified_text_clean)

tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198, 151652, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151653,   2610,
            525,    264,   5777,   6203,     13,  20205,    389,    279,   2701,
           5904,   4008,     11,   8253,    421,  24479,   1265,    387,  11676,
            624,  92602,     25,   1591,   4498,    279,  42002,    730,    579,
          22051,   3229,    

In [ ]:

copies[:,99:135] = 0
print(copies)
modified_text_clean = processor.tokenizer.decode(copies[0].cuda(), skip_special_tokens=True)
print("\nCleaner version:")
print(modified_text_clean)

tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198, 151652, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151653,   2610,
            525,    264,   5777,   6203,     13,  20205,    389,    279,   2701,
           5904,   4008,     11,   8253,    421,  24479,   1265,    387,  11676,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,    

In [ ]:
inputd = processor(
    text="<image>\n" + modified_text_clean ,           # Your text
    images=imaged ,  # Your masked image
    return_tensors="pt",
    padding=True,
).to("cuda")
print(inputd)
# Generate
outputs = model.generate(**inputd, return_dict_in_generate=True, output_scores=True)

{'input_ids': tensor([[   27,  1805,   397,  8948,   198,  2610,   525,   264, 10950, 17847,
           624,   872,   198,  2610,   525,   264,  5777,  6203,    13, 20205,
           389,   279,  2701,  5904,  4008,    11,  8253,   421, 24479,  1265,
           387, 11676, 50667, 50667, 50667, 50667, 17085, 42002,   594,  6981,
            13,  3197,   279, 42002,   594,  6981,  6963, 25527,   276,   504,
          1588,    11,   279,  1053, 14704,  3697,  2114,   323, 18822, 21023,
           311,  5505,  1435,    13,   576,  5535,  3166,   374,   429,  2474,
           220,    22,  8836,   389,   279,  2400,   315,   220,    17,    24,
            13,    16,    16,    13,    17,    15,    17,    15,    11,   279,
         79502,   315,   279, 42002,   594,  6981,   374,   537,  3881,    13,
           576,  1053, 14704,   374, 18822,  8098,   323,  5488,   429,  1052,
           686,   387, 10960, 15917,    13,   576, 42002,   374,  1083, 21023,
           311,   387,  7425,   553, 6

ValueError: Image features and image tokens do not match: tokens: 0, features 121

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": imaged},
            {"type": "text", "text": modified_text_clean}
        ]
    }
]
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputd = processor(
    text=text,
    images=imaged,
    return_tensors="pt",
    padding=True,
).to("cuda")

print(inputd)
outputs = model.generate(**inputd, return_dict_in_generate=True, output_scores=True,do_sample=False,       # Greedy decoding
    max_new_tokens=1,temperature=0.1)
print(outputs)

{'input_ids': tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198, 151652, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
         15165

In [ ]:
# Get all possible token IDs for yes/no
yes_tokens = []
no_tokens = []

for variant in ["yes", " yes", "Yes", " Yes", "YES", " YES"]:
    token_ids = processor.tokenizer.encode(variant, add_special_tokens=False)
    if token_ids:
        yes_tokens.append(token_ids[0])

for variant in ["no", " no", "No", " No", "NO", " NO"]:
    token_ids = processor.tokenizer.encode(variant, add_special_tokens=False)
    if token_ids:
        no_tokens.append(token_ids[0])

# Remove duplicates
yes_tokens = list(set(yes_tokens))
no_tokens = list(set(no_tokens))

print(f"Yes token IDs: {yes_tokens}")
print(f"No token IDs: {no_tokens}")

# Now get probabilities for ALL of them
answer_probs = torch.softmax(outputs.scores[0][0], dim=0)

yes_prob = sum([answer_probs[tid].item() for tid in yes_tokens])
no_prob = sum([answer_probs[tid].item() for tid in no_tokens])

print(f"Total Yes probability: {yes_prob:.4f}")
print(f"Total No probability: {no_prob:.4f}")

# Verify: what token was actually generated?
generated_token_id = generated_ids.sequences[0][-1].item()
print(f"\nActually generated token ID: {generated_token_id}")
print(f"Decoded: '{processor.tokenizer.decode([generated_token_id])}'")

Yes token IDs: [14080, 9834, 9454, 14004, 7414, 9693]
No token IDs: [5664, 2753, 2308, 8996, 902, 2152]
Total Yes probability: 0.1909
Total No probability: 0.7962

Actually generated token ID: 2753
Decoded: 'No'


In [ ]:
# split up the input_ids and the image_token_ids from x (containing both appended)
        input_ids = torch.tensor(x[:, :inputs["input_ids"].shape[1]])
        masked_image_token_ids = torch.tensor(
            x[:, inputs["input_ids"].shape[1]:])

        if use_cuda:
            input_ids = input_ids.cuda()
            masked_image_token_ids = masked_image_token_ids.cuda()

        # select / mask features and normalized boxes from masked_image_token_ids
        result = np.zeros(input_ids.shape[0])
        row_col = 16

        input = copy.deepcopy(inputs)
        # call the model for each "new image" generated with masked features
        for i in range(inputs['input_ids'].shape[0]):
            # here the actual masking of the image is happening. The custom masker only specified which patches to mask, but no actual masking has happened
            masked_text_inputs = inputs.copy()
            masked_text_inputs['input_ids'] = input_ids[i].unsqueeze(0)
            masked_image = copy.deepcopy(image)

            # pathify the image
            # torch.Size([1, 3, 384, 384]) image size ALBEF
            for k in range(masked_image_token_ids[i].shape[0]):
                if masked_image_token_ids[i][k] == 0:  # should be zero
                    m = k // patch_size  # 384 (img shape) / 16 (patch size)
                    n = k % patch_size
                    masked_image[:, :, m * patch_size:(m+1)*patch_size, n*patch_size:(n+1)*patch_size] = 0
            input["input_ids"] = input_ids
            input["pixel_values"] = masked_image
            outputs = model(**input,return_dict_in_generate=True,output_scores=True)
            # Get token IDs
            yes_id = processor.tokenizer.encode(" yes", add_special_tokens=False)[0]
            no_id = processor.tokenizer.encode(" no", add_special_tokens=False)[0]

            # Get probabilities for step 0
            step0_probs = torch.softmax(outputs.scores[0][0], dim=0)
            yes_prob = step0_probs[yes_id].item()
            no_prob = step0_probs[no_id].item()
            if label == 1:
                result[i] = yes_prob
            else:
                result[i] = no_prob